In [ ]:
## Run this cell if you are running your notebook in Google Colab!
## Make sure the data for this lesson is in the folder 'Digital Transformation Notebooks/Data'
## OR, change the location to the appropriate folder in your drive

## mount google colab
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/My Drive/Digital Transformation Notebooks/Data

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/My Drive/Digital Transformation Notebooks/Data


# Solving the Energy Systems Economic Dispatch Problem in Pyomo

When operating energy systems, we often have to decide how to allocate generation of power to multiple differerent resources in order to meet demand.  For example, we might have varied types of generation, including gas, nuclear, wind, hydropower, and solar, with different costs and operational constraints. We generally want to meet all energy demand while minimizing costs and satsifying the constraints of the system. This is an especially important challenge these days, with various renewables with intermittent generation, such as wind and solar, being added to the grid.

### The Economic Dispatch Problem

The economic dispatch problem minimizes the cost of meeting energy demands by the electrical grid subject to physical constraints on various power source systems (e.g., wind turbines, thermal generation, etc.) The formulation this notebook uses is adapted from [this source](https://jump.dev/JuMP.jl/stable/tutorials/applications/power_systems/), which provides a more in depth treatment of the problem if you are interested.

Suppose we have $n$ different power generators, with $g_i$ power output from the $i^{th}$ generator in MW (decision variables), and $c^g_i$  as the incremental cost (\$ per MWh) of running the generator. We also are able to inject a certain amount of wind power into the system: let $w$ be the wind power injected, and $c^w$  be the incremental cost (\$ per MWh) of this.

For constraints, we have minumum ($g^{min}_i$) and maximum ($g^{max}_i$) possible power outputs for each generator:

$$g^{min}_i \leq g_i \leq g^{max}_i$$.

We also have a constraint on windpower injection, where $w^f$ is the wind power forecasted for the time period in question:

$$0 \leq w \leq w_f$$

Finally, we must supply all forecasted demand $d_j$, such that the total power from wind injection and generation is equal to demand:

$$\sum_i g_i + w = d$$

We can formulate this problem mathematically as follows to minimize costs of energy distribution:

$$
\begin{aligned}
& \underset{g_i \in I, w}{\text{minimize}}
& & Z = \sum_{i \in I} c^g_i g_i  + c^w w \\
& \text{subject to}
& & g^{min}_i \leq g_i \leq g^{max}_i \: \forall \: i\\
& & & 0 \leq w \leq w_f \\
& & & \sum_i g_i + w = d
\end{aligned}
$$

First, install Pyomo and glpk if you are using Google Colab:

In [ ]:
!pip install -q pyomo
!apt-get install -y -qq glpk-utils

Assume we have data on the costs and restrictions of each generator. Let's load and view this data using Pandas:

In [ ]:
import pandas as pd
generator_data = pd.read_csv('generators.csv')
generator_data.head()

,Generator,Variable Cost $/MWh,Fixed Cost $,Min,Max
0,1,50,1000,0,1000
1,2,100,0,300,1000
2,3,75,100,0,900
3,4,65,100,5,1200


Now, let's load this generator data into variables. We'll also define variables for supplmentary data on our demand forecast, wind forecast, and cost of injecting wind.

In [ ]:
c_g = generator_data["Variable Cost $/MWh"].values
g_min = generator_data["Min"].values
g_max = generator_data["Max"].values
print("Wind table variable and column count:", generator_data.shape)

# demand forcast
demand = 1500.00
# wind forecast
w_forecast = 200.0
# wind cost
w_cost = 50.0


Wind table variable and column count: (4, 5)


Now, let's formulate this optimization problem in Pyomo. Our Objective function to minimize, $\sum_{i \in I} c^g_i g_i  + c^w w$, can be coded as `sum([c_g[i]*model.g[i] for i in I]) + w_cost*model.w[1]`. We similarly add our constraints as a Constraint List, call our solver, and print our results:

In [ ]:
from pyomo.environ import *

I = range(generator_data.shape[0])

# create a model
model = ConcreteModel()
#declare decision variables
model.g = Var(I,domain=NonNegativeReals)
model.w =  Var([1],domain=NonNegativeReals)
#declare objective
model.cost = Objective(expr = sum([c_g[i]*model.g[i] for i in I]) + w_cost*model.w[1], sense=minimize)
#declare constraints
model.cons = ConstraintList()
for i in I:
  model.cons.add(g_min[i] <= model.g[i])
  model.cons.add(model.g[i] <= g_max[i] )
model.cons.add(model.w[1] <= w_forecast)
model.cons.add(sum([model.g[i] for i in I]) + model.w[1] == demand)
#solve linear program
SolverFactory('glpk', executable='/usr/bin/glpsol').solve(model).write()
#display solution
print('\nCost = ', model.cost())
print('\nDecision Variables')
print('g = ', [model.g[i].value for i in I])
print('w = ', model.w[1].value)
print('Wind spill = ', w_forecast  - model.w[1].value)

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 90075.0
  Upper bound: 90075.0
  Number of objectives: 1
  Number of constraints: 10
  Number of variables: 5
  Number of nonzeros: 14
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 0
      Number of created subproblems: 0
  Error rc: 0
  Time: 0.0050182342529296875
# ----------------------------------------------------------
#   Solution Information
# -----------------------------

Looking at the optimal values, we can see that Generators 2, 3, and 4 are operating at their minimum levels, we might benefit if they could be run even less. Power generators might have a minimum power they can generate while they are on - can't we just turn one or more of them off? For this we would need a binary variable (0 or 1) for each generator, specifying if it was on or off. We will discuss such integer optimization problems in a future lesson.  

### Extensions

In practice, we would expect there to be a pretty significant temporal factor in these decisions. Wind and demand forecasts almost certainly vary significantly throughout the day. We could account for this by adding an additional index $t$ to all relevant variables and parameters: $u_{i,t}$, $g_{i,t}$, $w_{i,t}$, $w^f_{t}$,  $d_{t}$, but the problem would remain fundamentally the same.